# diagonal-via-strides — worked example 3: Extract diagonals of a batch of matrices via as_strided

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `diagonal-via-strides`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

For a contiguous `(B, N, N)` tensor, the batch stride is `N*N`, the row stride is `N`, and the column stride is `1`. To pull the main diagonal of every matrix at once, keep the batch dimension with stride `N*N` and add a diagonal dimension that walks with stride `N + 1`. The result is a `(B, N)` no-copy view.

## Worked solution

**Step 1 — understand the storage layout.** A contiguous `(B, N, N)` tensor lays out matrix 0 fully, then matrix 1, and so on. So advancing one matrix means jumping `N * N` elements: the batch stride is `N*N`. Within a matrix the row stride is `N` and the column stride is `1`.

**Step 2 — decide the output shape.** We want one diagonal per matrix: shape `(B, N)`. The first axis indexes the batch, the second walks the diagonal.

**Step 3 — assign a stride to each output axis.** Moving along axis 0 (next matrix) costs `N*N` in storage. Moving along axis 1 (next diagonal element) costs `N + 1`, exactly as in the single-matrix case (one row down + one col right). So `stride = (N*N, N + 1)`.

**Step 4 — offset is zero.** The very first element is `m[0, 0, 0]`, at linear offset `0`, so no `storage_offset` is needed.

**Step 5 — assemble and verify.** `m.as_strided(size=(B, N), stride=(N*N, N + 1))` gives the batched diagonals as a view. PyTorch's `torch.diagonal(m, dim1=1, dim2=2)` computes the same `(B, N)` result, so we assert equality against it and confirm storage is shared by writing through the view.

In [ ]:
def batched_diagonals(m: Tensor) -> Tensor:
    B, N, _ = m.shape
    return m.as_strided(size=(B, N), stride=(N * N, N + 1))


t.manual_seed(0)
B, N = 3, 4
m = t.arange(B * N * N).reshape(B, N, N)
d = batched_diagonals(m)
ref = t.diagonal(m, dim1=1, dim2=2)
print("batched diagonals shape:", tuple(d.shape))
print("matches torch.diagonal:", bool(t.equal(d, ref)))
print("first matrix diagonal:", d[0].tolist())